# Run a full evaluation pipeline end to end

Load SkySeaLand, configure the [`quality`](../reference/presets.md#quality), [`audit`](../reference/presets.md#audit)
and [`splits`](../reference/presets.md#splits) workflows in one YAML file, run the pipeline in a single call, inspect
the results, and execute the identical pipeline inside a container.

You are a model developer, data scientist, or T&E engineer who wants to run a complete evaluation pipeline from
configuration to containerized execution.

This tutorial demonstrates an end-to-end evaluation pipeline: dataset staging, configuration definition, pipeline
execution with `run_tasks()` or the command line, result inspection, and containerized execution. It is the first
tutorial in the learning path. If you are new to DataEval Flow, start with the {doc}`../home/quickstart`.
{doc}`metadata_triage` comes next.

## What you will do

1. Fetch SkySeaLand and export it in a layout that DataEval Flow supports.
2. Run [`triage`](../reference/presets.md#triage) to decide which factors to drop and how to bin the rest.
3. Define the datasets, sources, extractor, workflows and tasks in `end_to_end.yaml`.
4. Execute all tasks in a single call with `run_tasks()`.
5. Inspect the text reports, the HTML report, the structured findings and the flagged images.
6. Generate machine-readable result envelopes for downstream tools.
7. Execute the identical pipeline inside a container, without Python code.

The three tasks run different evaluation workflows:

| Task | Workflow | Sources | Extractor | Answers |
| --- | --- | --- | --- | --- |
| `clean_train` | `quality` | train | BoVW | Are training samples free of severe outliers and duplicates? |
| `audit_splits` | `audit` | train + test | BoVW | Is the data ready to train on, with no leakage between train and test? |
| `split_train` | `splits` | train | (none) | How should you partition training data into cross-validation folds? |

## What you will learn

- How to run `triage` to decide which factors to exclude and how to bin the rest.
- How to declare multiple evaluation workflows in a single YAML configuration file.
- How to run end-to-end pipelines using `run_tasks()` in Python.
- How to inspect text reports and HTML reports, and query structured findings programmatically.
- How to export auditable result envelopes for CI/CD gates.
- How to execute pipelines inside Docker without Python code.

## Prerequisites

- Install `dataeval-flow` (includes `dataeval`, `datamaite`, `pydantic`).
- Install `dataeval-plots` to visualize flagged images.
- Install `maite-datasets[datamaite]` to download and export SkySeaLand.
- Ensure network access for the initial download; subsequent runs read from disk.
- Install Docker if you plan to follow the "Run the same pipeline in Docker" section.

## Lay out the data root

The data root is the folder you run this notebook from. After a first run it holds:

```text
project/
├── end_to_end.yaml                    # read: the pipeline
├── data/
│   ├── skysealand/                    # downloaded on the first run; the exports are made from it
│   ├── skysealand_datamaite_train/    # exported on the first run; read: the train split
│   └── skysealand_datamaite_test/     # exported on the first run; read: the test split
├── cache/                             # written: embeddings and image statistics, reused on a re-run
└── output/end_to_end/                 # written: one .json result and one .html report per task
```

The dataset paths in `end_to_end.yaml` are relative to the data root, so `data/skysealand_datamaite_train` finds the
export. The notebook loads the YAML file in "Load and validate it", runs it in "Run the pipeline", and writes the
results in "Export the results". The "Run the same pipeline in Docker" section copies the YAML file and the two
exports into a fresh root and mounts it read-only in the container, which writes to `/output` and `/cache`. The triage
run in "Analyze the metadata" builds its pipeline in Python. The notebook does not read the `val` split.

## Load the dataset

[SkySeaLand](https://www.kaggle.com/datasets/mdzahidhasanriad/skysealand) is an overhead object-detection dataset with
1,307 frames from four sites and 19,102 annotations across `airplane`, `boat`, `car`, and `ship`. The dataset provides
`train` (1,048 frames), `val` (132), and `test` (127) splits. This pipeline evaluates `train` and `test`.

Use `maite-datasets` with `as_datamaite=True` to export the dataset into standard COCO format for DataEval Flow. Each
export directory is named by dataset and `image_set`, which keeps the splits isolated on disk.

In [ ]:
from pathlib import Path

from maite_datasets.object_detection import SkySeaLand

data_root = Path("./data")

# Download once into ./data/skysealand; exports reuse the cached download.
SkySeaLand(root=data_root, image_set="base", download=True)

split_paths = {name: data_root / f"skysealand_datamaite_{name}" for name in ("train", "test")}
for image_set in split_paths:
    SkySeaLand(root=data_root, image_set=image_set, as_datamaite=True)

print("\n".join(f"{name}: {path}" for name, path in split_paths.items()))

### Confirm it loads

Before you write the configuration, verify that DataEval Flow can read the exported files. `load_dataset()` uses the
same loader as the pipeline runtime.

In [ ]:
from dataeval_flow import load_dataset

train_ds = load_dataset(split_paths["train"], dataset_format="coco")
image, target, _ = train_ds[0]

print(f"Images:      {len(train_ds)}")
print(f"Image shape: {image.shape}")
print(f"Boxes:       {len(target.boxes)}")
print(f"Classes:     {train_ds.metadata['index2label']}")

## Analyze the metadata

Before you commit to a `metadata:` policy, run the `triage` preset. It flags factors that carry no information and
factors whose bin counts would otherwise be derived silently (and unstably) at run time.

This section covers only the operational checkpoint: run `triage`, act on its findings, and move on.
{doc}`metadata_triage` walks through reading triage findings, distribution charts, and remediation policies.

In [ ]:
from dataeval_flow import run_task
from dataeval_flow.config import (
    CocoDatasetConfig,
    MetadataPolicyConfig,
    PipelineConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.workflows.triage import TriageConfig

triage_config = PipelineConfig(
    max_processes=2,
    metadata=[MetadataPolicyConfig(name="skysealand_factors", intrinsic_factors=["visual", "pixel"])],
    datasets=[CocoDatasetConfig(name="skysealand_train", path=str(split_paths["train"]))],
    views=[
        ViewConfig(
            name="sample300",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 0}),
                ViewOperation(type="Limit", params={"size": 300}),
            ],
        )
    ],
    sources=[SourceConfig(name="train_src", dataset="skysealand_train", view="sample300")],
    workflows=[TriageConfig(name="triage", metadata="skysealand_factors")],
    tasks=[TaskConfig(name="triage_train", workflow="triage", sources="train_src")],
)

triage_result = run_task(triage_config, "triage_train", data_dir=Path("."), cache_dir=Path("./cache"))

In [ ]:
triage = triage_result.steps["factor-triage"].output.data()
findings = triage["findings"]
degenerate = sorted({f.factor for f in findings if f.category == "degenerate"})
unbinned = sorted({f.factor for f in findings if f.category == "unbinned"})

print(f"Factors:              {triage['factor_count']}")
print(f"Findings:             {len(findings)} ({triage['counts'].get('blocking', 0)} blocking)")
print(f"Degenerate (exclude): {degenerate}")
print(f"Need explicit bins:   {len(unbinned)} factors")

`triage` flags `label_file_exists`, `instance_missing`, and `unit_missing` as degenerate. Every frame holds the same
value for each, so the factor separates nothing. It also flags `instance_zeros` for a sentinel-value remap before it
can be binned. That judgment call is covered in {doc}`metadata_triage`, so this pipeline excludes the factor instead.
The remaining continuous pixel and visual factors need explicit bin counts. Without them, `audit` derives the counts
from whichever sample happens to run, and warns that it did.

The `metadata:` section of `end_to_end.yaml` already applies these findings. It excludes the four degenerate and
unresolved factors, and every remaining continuous factor has a pinned `continuous_factor_bins` count.
{doc}`../how_to/configure_metadata_binning` explains how to declare cuts and vocabularies.

## Set up the configuration

Define pipeline execution in a single YAML file, in modular sections:

| Section | What it declares |
| --- | --- |
| `datasets` | Where data lives and how to read it |
| `views` | How to narrow a dataset (limit, index range, class filter) |
| `sources` | A dataset plus an optional view: the input consumed by tasks |
| `extractors` | How embeddings are produced |
| `workflows` | Named parameter sets for evaluations |
| `tasks` | Binds a workflow to sources (and an extractor when needed) |

Print the configuration file `end_to_end.yaml`:

In [ ]:
config_path = Path("end_to_end.yaml")
print(config_path.read_text())

### Load and validate it

Call `load_config()` to parse the YAML file and validate it against the pipeline schema. Validation catches invalid
parameters, unknown workflow types, and missing references before execution.

In [ ]:
from dataeval_flow import load_config

config = load_config(config_path)

print(f"Datasets:   {[d.name for d in config.datasets]}")
print(f"Sources:    {[s.name for s in config.sources]}")
print(f"Extractors: {[e.name for e in config.extractors]}")
print(f"Workflows:  {[(w.name, w.type) for w in config.workflows]}")
print(f"Tasks:      {[(t.name, t.workflow, t.sources) for t in config.tasks]}")

## Run the pipeline

Call `run_tasks()` to execute all enabled tasks. It returns each task's result keyed by task name, in execution order.
Two arguments matter here:

- `data_dir` is the base directory for resolving relative paths in the configuration.
- `cache_dir` is the directory for caching embeddings, image statistics, and metadata across tasks and runs.

Tasks that share datasets reuse the intermediate calculations stored in the cache.
{doc}`../how_to/reuse_results_with_cache` explains caching behaviors and invalidation.

In [ ]:
from dataeval_flow import run_tasks

results = run_tasks(config, data_dir=Path("."), cache_dir=Path("./cache"))

In [ ]:
for name, result in results.items():
    status = "OK " if result.success else "FAIL"
    warnings = result.warning_count
    elapsed = result.metadata.execution_time_s or 0.0
    print(f"[{status}] {name:<16} {result.type:<16} {elapsed:>6.1f}s  {warnings} warning(s)")
    for error in result.errors:
        print(f"         {error}")

In [ ]:
assert all(r.success for r in results.values()), [r.errors for r in results.values() if not r.success]

In [ ]:
clean_result, audit_result, split_result = results["clean_train"], results["audit_splits"], results["split_train"]

## Display the results

Each task result exposes three primary interfaces:

- `result.report()` returns a formatted text summary.
- `result.findings` holds the structured finding objects.
- `result.steps` holds the numbers behind the findings. All three workflows run as chains of steps, so their results
  hold each step's output in `result.steps`, by step name. The split's part indices are in
  `result.steps["split"].details["indices"]`, and the audit's verdict is `result.verdict`.

Call `report(detailed=False)` for high-level summaries, or `report(detailed=True)` for per-finding breakdowns.

### Read the summary reports

In [ ]:
for result in results.values():
    print(result.report(detailed=False))

### Read the HTML report

`to_html()` draws the same blocks as the text report, as one self-contained page. The cell below shows the audit
task's page. {doc}`view_html_reports` shows what else the page does.

In [ ]:
# The page styles its whole document, so the docs show it in an iframe of its own.
import html

from IPython.display import display


def show(page: str, height: int = 560) -> None:
    """*page* in an iframe, so its stylesheet and script stay inside it."""
    frame = (
        f'<iframe srcdoc="{html.escape(page, quote=True)}" title="HTML report" loading="lazy" '
        f'style="width:100%;height:{height}px;border:1px solid #d1d9e0;border-radius:6px"></iframe>'
    )
    display({"text/html": frame}, raw=True)

In [ ]:
show(audit_result.to_html())

### Query the structured findings

Each finding has a `title`, a `severity` (`ok`, `info`, or `warning`), a short `brief`, a `description` (`None` where
the brief says it all), and its evidence as report `blocks`. You can query these programmatically in automated CI/CD
gates.

In [ ]:
for result in results.values():
    print(f"\n{result.type}")
    for finding in result.findings:
        marker = {"warning": "[!!]", "ok": "[ok]"}.get(finding.severity, "[..]")
        print(f"  {marker} {finding.title:<34} {finding.brief or ''}")

### Inspect the flagged images

The cleaning report summarizes the count of flagged images. Retrieve specific sample indices from the `outliers` and
`duplicates` steps, whose outputs are DataEval's own, and slice `result.sources` directly without reloading data. It
holds each source the task read, after its view.

In [ ]:
issues = clean_result.steps["outliers"].output.data()
image_issues = issues.filter(issues["target_index"].is_null())  # the rest flag single boxes
outlier_indices = sorted(set(image_issues["item_index"].to_list()))
near_groups = clean_result.steps["duplicates"].output.items.near

print(f"Image outliers:        {len(outlier_indices)} images, {image_issues.height} flags")
print(f"Near-duplicate groups: {len(near_groups)}")

In [ ]:
from dataeval_plots import plot

if outlier_indices:
    _ = plot(
        clean_result.sources["train_src"],
        indices=outlier_indices[:6],
        images_per_row=3,
        figsize=(12, 8),
        show_labels=True,
    )

### Read the audit verdict

The audit task judges train and test together. Its verdict says whether the data is ready to train on. A warning from
a blocking check, such as leakage between the splits, makes it not ready, and any other warning, or a check it could
not assess, is a caveat. The task names the BoVW extractor, so every check is assessed. Train and test share no image,
and test has no class train lacks, so nothing blocks. The verdict is ready with caveats: the outlier, metadata,
coverage and shortcut warnings. {doc}`audit` reads a verdict in depth and keeps a record of what was audited.

In [ ]:
verdict = audit_result.verdict
assert verdict is not None

print(verdict.line())
for item in verdict.blocking + verdict.warnings:
    print(f"  [!!] {item.step:<32} {item.brief}")
for item in verdict.not_assessed:
    print(f"  [..] {item.step:<32} not assessed: {item.reason}")

### Read the split indices

The splitting task generates explicit index lists for each part, in the split step's details. With `folds: 1`, as
here, `train`, `val` and `test` are each one list; with more folds, `train` and `val` are keyed by fold. Use these
indices to construct PyTorch `Subset` or `DataLoader` instances. {doc}`dataset_splitting` covers stratification,
cross-validation folds, and group-aware splitting.

In [ ]:
indices = split_result.steps["split"].details["indices"]

print(f"Split sizes: { {part: len(idx) for part, idx in indices.items()} }")
print(f"Train indices (first 10): {indices['train'][:10]}")
print(f"Val   indices (first 10): {indices['val'][:10]}")
print(f"Test  indices (first 10): {indices['test'][:10]}")

## Export the results

Call `export()` to write the result envelope to disk. A result envelope holds the findings alongside execution
metadata: timestamps, tool versions, dataset identifiers, and fully resolved configurations. `to_html()` renders the
same report as one self-contained page that you can open in a browser, attach to a ticket, or print to PDF. The cell
below writes both for each task.

In [ ]:
output_dir = Path("./output/end_to_end")

for name, result in results.items():
    written = result.export(output_dir / f"{name}.json")
    page = output_dir / f"{name}.html"
    page.write_text(result.to_html(), encoding="utf-8")
    print(f"{written}  ({written.stat().st_size:,} bytes)")
    print(f"{page}  ({page.stat().st_size:,} bytes)")

In [ ]:
import json

envelope = json.loads((output_dir / "split_train.json").read_text())

print(f"Top-level keys: {list(envelope)}")
print(f"Tool:           {envelope['metadata']['tool']} {envelope['metadata']['tool_version']}")
print(f"Timestamp:      {envelope['metadata']['timestamp']}")
print(f"Sources:        {envelope['metadata']['source_descriptions']}")

## Run the same pipeline in Docker

Run the same YAML pipeline in Docker without Python dependencies. The container consumes `end_to_end.yaml`, reads
datasets from `/dataeval`, and writes result envelopes to `/output`. To submit this same `end_to_end.yaml` over HTTP
to a long-running service instead, and follow the run to its results, see {doc}`../how_to/run_flow_as_a_service`.

The container uses three mount paths:

| Mount | Mode | Holds |
| --- | --- | --- |
| `/dataeval` | read-only | Data root: datasets, models, configuration files |
| `/output` | read-write | Reports and result envelopes |
| `/cache` | read-write | Caches embeddings and statistics across runs |

### Lay out the workspace

Create a root directory containing `end_to_end.yaml` and the exported dataset splits:

```bash
mkdir -p dataeval-run/data dataeval-run/output dataeval-run/cache
cp -r docs/source/notebooks/data/skysealand_datamaite_train dataeval-run/data/
cp -r docs/source/notebooks/data/skysealand_datamaite_test dataeval-run/data/
cp docs/source/notebooks/end_to_end.yaml dataeval-run/

tree -L 3 dataeval-run
```

### Get the image

Pull the pre-built container image or build it locally:

```bash
# Pre-built (cpu / cu126 / cu130)
docker pull harbor.jatic.net/aria/dataeval-flow:latest-cpu

# Optional: verify the signature
cosign verify --key docker/cosign.pub harbor.jatic.net/aria/dataeval-flow:latest-cpu

# Or build locally from a checkout
docker build -f docker/Dockerfile.cpu -t dataeval-flow:cpu .
```

### Run it

Use `--user` so that `/output` and `/cache` remain writable by your host account. The console prints each report's
short form, and `-v` prints the full reports instead.

```bash
cd dataeval-run

docker run --rm \
    --user "$(id -u):$(id -g)" \
    --mount type=bind,source="$PWD",target=/dataeval,readonly \
    --mount type=bind,source="$PWD/output",target=/output \
    --mount type=bind,source="$PWD/cache",target=/cache \
    harbor.jatic.net/aria/dataeval-flow:latest-cpu \
    python -m dataeval_flow --config end_to_end.yaml -v
```

On GPU hosts, include `--gpus all` and select a CUDA image:

```bash
docker run --rm --gpus all \
    --user "$(id -u):$(id -g)" \
    --mount type=bind,source="$PWD",target=/dataeval,readonly \
    --mount type=bind,source="$PWD/output",target=/output \
    --mount type=bind,source="$PWD/cache",target=/cache \
    harbor.jatic.net/aria/dataeval-flow:latest-cu130 \
    python -m dataeval_flow --config end_to_end.yaml -v
```

:::{note}
Specify `--config` explicitly when running with multiple YAML files in your data root.
Without `--config`, DataEval Flow merges all configuration files in the root directory.
:::

### Read the output

The container produces `result.json`, `result.txt`, and run logs in `/output`:

```bash
find output -type f
# output/result.log
# output/results/result.json
# output/results/result.txt
```

Query the generated results with `jq`:

```bash
# Inspect executed tasks
jq -r 'keys[]' output/results/result.json

# Print findings and severities: each workflow lists its findings at the top of its entry
jq -r 'to_entries[] | .key as $task | .value.findings[]
       | "\($task)\t\(.severity)\t\(.title)"' output/results/result.json

# Gate CI/CD pipelines on warnings
jq -e '[.[].health.warnings] | add == 0' \
    output/results/result.json > /dev/null \
    && echo "PASS: no warnings" || echo "FAIL: warnings present"

# Inspect split partition sizes
jq -c '.split_train.steps.split.details.indices | map_values(length)' output/results/result.json
```

### Run offline

Once the container images and datasets are staged locally, you can run entirely offline:

```bash
docker run --rm --network none \
    --user "$(id -u):$(id -g)" \
    -e HF_HUB_OFFLINE=1 -e HF_DATASETS_OFFLINE=1 \
    --mount type=bind,source="$PWD",target=/dataeval,readonly \
    --mount type=bind,source="$PWD/output",target=/output \
    --mount type=bind,source="$PWD/cache",target=/cache \
    harbor.jatic.net/aria/dataeval-flow:latest-cpu \
    python -m dataeval_flow --config end_to_end.yaml -v
```

## Conclusion

In this tutorial, you learned how to:

- Stage dataset splits in isolated disk directories.
- Run `triage` to decide which factors to exclude and how to bin the rest.
- Define multi-workflow pipelines in a single YAML configuration file.
- Execute pipelines using `run_tasks()` and the container CLI.
- Inspect formatted reports and extract structured findings.
- Export auditable result envelopes for CI/CD gates and downstream tools.
- Execute the complete evaluation pipeline inside Docker.

## Next steps

- {doc}`metadata_triage` — Read triage findings, distribution charts, and remediation policies.
- {doc}`data_cleaning` — Detect outliers and duplicates in a dataset.
- {doc}`../how_to/containerized_workflows` — Use the Docker execution options and flags.
- {doc}`../how_to/read_evaluation_outputs` — Query the result envelope structure.

## On your own

To repeat the pipeline with your own data and settings, change these entries in `end_to_end.yaml`:

- Point the `path` of each entry under `datasets` at your own data, and set its `format`. Keep one directory for each
  split, because a COCO export records no split of its own.
- To audit the whole training set, remove the `view` from the `train_src` source. To change the sample, edit the
  `size` of the `Limit` operation in the `sample300` view.
- To change the factors the analysis reads, edit `exclude` and `continuous_factor_bins` in the `metadata` entry. Run
  `triage` on your data to find both.
- To change what counts as an outlier, edit `outlier_threshold` and `flags` in the `clean` and `audit` workflows. To
  change when the cleaning findings warn, edit the entries under `checks` in the `clean` workflow.
- To split differently, edit `folds`, `test_frac`, `val_frac`, and `stratify` in the `split` workflow. `val_frac`
  applies with `folds: 1` only, and unset is 0.1.
- To change the embeddings, edit `vocab_size` or `batch_size` of `bovw_ext`, or replace it with another entry under
  `extractors`, such as one with `model: onnx` and a `model_path`.
- To change the seed that pins clustering and splitting, edit `seed` at the top of the file.